# Introduction
The following program was authored by Max Tejera in 2023-2024. The program was written to conduct analysis on the impacts of the Colorado Equal Pay for Equal Work Act (EPEWA) on the gender wage gap in Colorado. The original analysis was for Max's master of economics capstone research project at the University of Colorado Denver during the fall term of 2023. The code was originally authored in Stata, but has been translated to Python here with some additional functionality.

**Update (October 2026):** The analysis was refreshed in October 2026 to extend the sample from May 2024 through September 2026. The hardcoded file paths, sample names, minimum wage tables and industry crosswalk were replaced with code that reads them from current sources. The original July 2024 window can still be reproduced by setting `SAMPLE_END = (2024, 5)` in the configuration cell. Results for both windows are saved side by side for comparison.

# Section 0: Setup
The first block imports the libraries used in the project. The second is a configuration cell. It sets:

- the data folder and the location of the IPUMS API key, read from the environment variables `EPEWA_DATA_DIR` and `IPUMS_API_KEY_FILE` so no personal paths or keys are stored in the notebook;
- the sample window (`SAMPLE_START`, `SAMPLE_END`);
- `SUBMIT_NEW_EXTRACT`, which should be `True` only when a new IPUMS extract is needed, so that re-running the notebook never files duplicate requests;
- a connection setting for running behind RAND's network (see the README)
- the helper functions used for regressions and charts.

In [1]:
import bisect
import gzip
import os
import re
import shutil
import ssl
import warnings
from datetime import datetime
from pathlib import Path
import certifi
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
import requests
import statsmodels.formula.api as smf
from bs4 import BeautifulSoup
from ipumspy import IpumsApiClient, MicrodataExtract, readers
from requests.adapters import HTTPAdapter

In [2]:
# ---- Project configuration -------------------------------------------------
# Paths come from environment variables so the notebook runs on any machine.
DATA_DIR = Path(os.environ.get("EPEWA_DATA_DIR", Path.cwd() / "data"))
IPUMS_API_KEY_FILE = os.environ.get("IPUMS_API_KEY_FILE")   # text file holding the key (or set IPUMS_API_KEY)

# Study window for the IPUMS CPS monthly samples.
# (2024, 5) reproduces the July 2024 thesis run; set to the latest month to refresh.
SAMPLE_START = (2010, 1)
SAMPLE_END   = (2026, 9)

# Set True only when you need a fresh IPUMS extract (new sample window or new data release).
# False reuses the newest extract already in DATA_DIR, so "Run All" never files duplicate requests.
SUBMIT_NEW_EXTRACT = False

DATA_DIR.mkdir(parents=True, exist_ok=True)
download_dir = DATA_DIR

In [3]:
# ---- TLS workaround for RAND's Cisco Secure Access proxy ---------------------------------------------
# Behind RAND's network, HTTPS is re-signed by Cisco's root. Python 3.13+ also demands an Authority Key
# Identifier on every certificate, which that chain lacks. This keeps full certificate verification
# (against the certifi bundle that has RAND_PKI_Root.pem appended) and only drops that one extra strict flag.


class RelaxedStrictAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context(cafile=certifi.where())
        ctx.verify_flags &= ~ssl.VERIFY_X509_STRICT
        kwargs['ssl_context'] = ctx
        return super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount('https://', RelaxedStrictAdapter())
requests.get = _session.get          # every requests.get in the notebook now uses this session

### Helper functions
Reusable functions used by the later sections: `fit_did` and `run_by_group` fit the difference-in-differences regression for one group or many, `term_table` prints selected coefficients, and `plot_group_gaps` and `plot_state_bars` draw the charts.

In [4]:
# ---- Regression helpers ---------------------------------------------------
KEY_FEMALE = 'C(female)[T.1]'
KEY_INTERACTION = 'C(female)[T.1]:C(epewa)[T.1]'

FORMULA_NO_IND = 'log_earnyear ~ C(female) + C(epewa) + C(female):C(epewa) + C(age) + C(race_cat) + C(educ)'
FORMULA_IND = FORMULA_NO_IND + ' + C(ind_cat20_label)'

def fit_did(data, formula):
    """OLS with HC3 robust standard errors. Returns the model and the key coefficients."""
    with warnings.catch_warnings():
        # Subsets (one state, one industry) can be missing some category levels; statsmodels then reports a
        # rank-deficient design matrix. The coefficients of interest are still estimated, so the warning is silenced.
        warnings.filterwarnings('ignore', message='The design matrix is rank-deficient')
        model = smf.ols(formula, data=data).fit(cov_type='HC3')
    return model, {
        'female_coef': model.params[KEY_FEMALE],
        'female_pval': model.pvalues[KEY_FEMALE],
        'interaction_coef': model.params[KEY_INTERACTION],
        'interaction_pval': model.pvalues[KEY_INTERACTION],
    }

def run_by_group(df, group_col, formula, min_obs=2, include_national=False, add_gaps=True):
    """Fit the model separately for each group (optionally plus one row for the whole sample).
    Returns {group: model} and a results table. With add_gaps=True (log earnings), it also adds the
    'cents on the dollar' pre- and post-law gaps."""
    models, rows = {}, {}
    for group, subset in df.groupby(group_col, observed=True):
        if len(subset) >= min_obs:
            models[group], rows[group] = fit_did(subset, formula)
    if include_national:
        models['United States'], rows['United States'] = fit_did(df, formula)
    out = pd.DataFrame.from_dict(rows, orient='index')
    if add_gaps:
        out['pre_epewa_wage_gap'] = 1 + out['female_coef']
        out['post_epewa_wage_gap'] = out['pre_epewa_wage_gap'] + out['interaction_coef']
    return models, out

def term_table(model, patterns):
    """Coefficient, standard error and p-value for the model terms whose names contain any of the given strings."""
    names = [t for t in model.params.index if any(p in t for p in patterns)]
    return pd.DataFrame({'coef': model.params[names], 'std_err': model.bse[names], 'p_value': model.pvalues[names]}).round(4)

In [5]:
# ---- Chart helpers (dark theme), used by Sections 4 and 5 -------------------
# Period labels come from the config cell, so they update when SAMPLE_END changes.
PRE_PERIOD = "Jan 2010 - Dec 2020"
POST_PERIOD = f"Jan 2021 - {pd.Timestamp(SAMPLE_END[0], SAMPLE_END[1], 1):%b %Y}"

# Every chart is also saved as a standalone HTML file here (GitHub does not display Plotly charts inside notebooks).
CHART_DIR = DATA_DIR / 'results' / f"{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}_{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}" / 'charts'

# Highlighted jurisdictions get a fixed accent color; every other state is colored by its value.
HIGHLIGHT_COLORS = {'Colorado': '#34d399', 'California': '#fbbf24', 'Washington': '#60a5fa', 'United States': '#f472b6'}
GROUP_COLORS = {**HIGHLIGHT_COLORS, 'Rest of US': '#94a3b8'}   # Section 4 comparison groups
BAR_SCALE = ['#334155', '#3b6ea5', '#22d3ee']                  # slate -> blue -> cyan, low to high value
BG_TOP, BG_BOTTOM = (8, 12, 22), (30, 41, 59)                   # background gradient, top to bottom (RGB)
FONT = "Bahnschrift, 'Segoe UI Variable Display', 'Segoe UI', Helvetica, Arial, sans-serif"   # Bahnschrift ships with Windows 10/11
ROW_HEIGHT = 22                                                 # pixels per state, so every state fits


def _save_chart(fig, name):
    """Write the figure to CHART_DIR/<name>.html (small file; loads Plotly's JavaScript from the web)."""
    CHART_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(CHART_DIR / f"{name}.html", include_plotlyjs='cdn', config={'displayModeBar': False})


def _gradient_bands(top, bottom, n=48):
    """Fake a vertical gradient (Plotly has no native gradient background). Each rectangle runs from the
    bottom edge up to its own top edge and is painted over the one before it, so there are no visible seams."""
    bands = []
    for i in range(n):
        t = i / (n - 1)
        rgb = tuple(round(a + (b - a) * t) for a, b in zip(top, bottom))
        bands.append(dict(type='rect', xref='paper', yref='paper', x0=0, x1=1,
                          y0=0, y1=1 - i / n, fillcolor=f"rgb{rgb}", line_width=0, layer='below'))
    return bands


def _hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{alpha})"


def _apply_dark_theme(fig, headline, controls, period, height, footnote):
    """Shared look: title, fonts, gradient background, footnote."""
    fig.update_layout(
        title=dict(text=f"<b>{headline}</b><br><span style='font-size:14px;color:#94a3b8'>Controls: {controls}"
                        f"<br>{period}</span>", x=0.5, xanchor='center', y=1, yanchor='top', pad=dict(t=40),
                   font=dict(size=24, color='#f8fafc')),
        font=dict(family=FONT, color='#cbd5e1'),
        paper_bgcolor='#0b1220', plot_bgcolor='rgba(0,0,0,0)', shapes=_gradient_bands(BG_TOP, BG_BOTTOM),
        height=height, margin=dict(l=20, r=60, t=170, b=120))
    fig.add_annotation(text=footnote, x=0.5, y=0, yanchor='top', yshift=-70, xref='paper', yref='paper',
                       showarrow=False, font=dict(size=12, color='#94a3b8'))


def plot_state_bars(results, value_col, pval_col, headline, controls, period, x_range, save_as=None):
    """Section 5: horizontal bar chart of one value per state, sorted ascending. A state gets an asterisk
    when its p-value (pval_col) is above 0.10. If save_as is given, the chart is also saved as HTML."""
    d = results.sort_values(value_col, ascending=True)
    spread = d[value_col].max() - d[value_col].min()
    norm = ((d[value_col] - d[value_col].min()) / spread) if spread else d[value_col] * 0 + 0.5
    base = pc.sample_colorscale(BAR_SCALE, list(norm))
    colors = [HIGHLIGHT_COLORS.get(state, c) for state, c in zip(d.index, base)]
    outline = ['#f8fafc' if state in HIGHLIGHT_COLORS else 'rgba(0,0,0,0)' for state in d.index]
    y_labels = [f"<b>{state}{'*' if pval > 0.10 else ''}</b>" for state, pval in zip(d.index, d[pval_col])]
    text = [f"<b>{v * 100:,.1f}%</b>" for v in d[value_col]]

    fig = go.Figure(go.Bar(
        x=d[value_col], y=y_labels, orientation='h', text=text, textposition='outside',
        textfont=dict(color='#f1f5f9', size=12),
        marker=dict(color=colors, line=dict(color=outline, width=1.5)),
        hovertemplate='%{y}: %{text}<extra></extra>', cliponaxis=False))
    _apply_dark_theme(fig, headline, controls, period, 260 + ROW_HEIGHT * len(d),
                      "*Not statistically significant at the 10% level (p > 0.10)")
    x_dtick = 0.02 if (x_range[1] - x_range[0]) <= 0.2 else 0.05   # fixed tick spacing so labels never collide
    fig.update_layout(
        bargap=0.25,
        xaxis=dict(range=x_range, tickformat='.0%', dtick=x_dtick, tickangle=0, showgrid=True,
                   gridcolor='rgba(148,163,184,0.15)', zeroline=True, zerolinecolor='rgba(148,163,184,0.45)',
                   showline=False, tickfont=dict(size=13, color='#cbd5e1', weight='bold')),
        yaxis=dict(type='category', dtick=1, showgrid=False, showline=False,
                   tickfont=dict(size=12, color='#f1f5f9'), automargin=True))
    if save_as:
        _save_chart(fig, save_as)
    fig.show()
    return fig


def plot_group_gaps(results, headline, controls, save_as=None):
    """Section 4: pre- and post-law wage gap for each comparison group, sorted by the size of the change.
    Faded bar = before the law, solid bar = after. The boxed number is the change in percentage points.
    If save_as is given, the chart is also saved as HTML."""
    assert len(results) < 10, "results_df holds the state table. Run the Section 4 regression cell directly above this chart first."
    d = results.copy()
    d['pre_epewa_wage_gap'] = 1 + d['female_coef']
    d['post_epewa_wage_gap'] = d['pre_epewa_wage_gap'] + d['interaction_coef']
    d = d.sort_values('interaction_coef', ascending=False)
    groups = list(d.index)
    accent = [GROUP_COLORS.get(g, '#94a3b8') for g in groups]

    def bars(name, col, colors):
        return go.Bar(name=name, x=[f"<b>{g}</b>" for g in groups], y=d[col], marker=dict(color=colors, line_width=0),
                      text=[f"<b>{v * 100:,.1f}%</b>" for v in d[col]], textposition='outside',
                      textfont=dict(color='#f1f5f9', size=13), cliponaxis=False,
                      hovertemplate=f'%{{x}} ({name}): %{{text}}<extra></extra>')

    fig = go.Figure([bars('Before the law', 'pre_epewa_wage_gap', [_hex_to_rgba(c, 0.45) for c in accent]),
                     bars('After the law', 'post_epewa_wage_gap', accent)])
    _apply_dark_theme(fig, headline, controls, f"{PRE_PERIOD} vs. {POST_PERIOD}", 720,
                      "Faded bars: before the law  |  Solid bars: after the law  |  Boxed value: change in percentage points")
    for g, c, change in zip(groups, accent, d['interaction_coef']):
        fig.add_annotation(x=f"<b>{g}</b>", y=0.975, xref='x', yref='y', text=f"<b>{change * 100:+.1f} pts</b>",
                           showarrow=False, font=dict(size=14, color=c), bgcolor='rgba(11,18,32,0.85)',
                           bordercolor=c, borderwidth=1.5, borderpad=5)
    fig.update_layout(
        barmode='group', bargap=0.35, bargroupgap=0.08, showlegend=False,
        xaxis=dict(showgrid=False, tickfont=dict(size=14, color='#f1f5f9')),
        yaxis=dict(range=[0.7, 1], tickformat='.0%', dtick=0.05, showgrid=True, gridcolor='rgba(148,163,184,0.15)',
                   showline=False, tickfont=dict(size=13, color='#cbd5e1', weight='bold')))
    if save_as:
        _save_chart(fig, save_as)
    fig.show()
    return fig

# Section 1: Load Current Population Survey Data via API
This section uses the IPUMSPY library from IPUMS and an API key saved outside the project folder to load the selected sample from the Current Population Survey (CPS) into a Pandas dataframe.

The selected sample is every monthly sample from January 2010 through September 2026 (200 samples). IPUMS has no October 2025 sample because those data were not collected during the federal government shutdown. Later on the sample will be divided into two time periods by the EPEWA variable (Equal Pay for Equal Work Act). EPEWA was passed during the Colorado 2019 legislative session and came into effect January 2021. To create a before and after comparison, the pre-law period was defined as January 2010 to December 2020. This period was chosen because the U.S. economy was mostly stable during this time, with some major abberations in the latter 10 months of 2020. This pre-law period is intended to capture a pre-trend in the gender wage gap prior to EPEWA coming into effect. The larger sample of 2010-2019 period likely offsets any major disruptions from the pandemic.

The CPS variables selected are elaborated on in the later sections. The number of variables selected has been minimized to optimize loading time. The first load for a given sample window takes approximately 30 minutes. I believe this is due to how IPUMSPY maps the data while loading. The filtered data are then cached to disk under a name that includes the sample window, so later runs load in seconds.

The sample names are pulled from the IPUMS API and filtered to the study window set in the configuration cell. A check lists any months missing from the window.

IPUMS requires all researchers to cite them when using their data. They are cited below and within the code.

##### Citation
> Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Etienne Breton, Grace Cooper, Julia A. Rivera Drew, Stephanie Richards, David Van Riper, and Kari C.W. Williams. IPUMS CPS: Version 13.0 [dataset]. Minneapolis, MN: IPUMS, 2025. https://doi.org/10.18128/D030.V13.0

In [6]:
# Open IPUMS API. The key is never stored in the repo.
api_key = os.environ.get("IPUMS_API_KEY")
if not api_key:
    if not IPUMS_API_KEY_FILE:
        raise RuntimeError("Set the IPUMS_API_KEY or IPUMS_API_KEY_FILE environment variable (see README).")
    api_key = Path(IPUMS_API_KEY_FILE).read_text().strip()
ipums = IpumsApiClient(api_key)

In [7]:
# Get the CPS monthly sample IDs directly from the IPUMS API
# (replaces the BeautifulSoup scrape of the sample_ids web page).
# The study window (SAMPLE_START / SAMPLE_END) is set in the config cell.
all_samples = ipums.get_all_sample_info("cps")   # {sample_id: description}

pattern = re.compile(r"^cps(\d{4})_(\d{2})[a-z]$")   # e.g. cps2010_01s

filtered_sample_dict = {}
for sample_id, description in all_samples.items():
    m = pattern.match(sample_id)
    if not m:
        continue
    if "ASEC" in description:   # skip the annual supplement samples
        continue
    ym = (int(m.group(1)), int(m.group(2)))
    if SAMPLE_START <= ym <= SAMPLE_END:
        filtered_sample_dict[sample_id] = description

CpsSamples = sorted(filtered_sample_dict, key=lambda s: (s[3:7], s[8:10]))
assert CpsSamples, "No samples found. Check the filters or the IPUMS API connection."
print(f"{len(CpsSamples)} samples: {CpsSamples[0][8:10]}/{CpsSamples[0][3:7]} to {CpsSamples[-1][8:10]}/{CpsSamples[-1][3:7]}")

# Flag any missing months in the window so gaps are not silent
expected, (y, mth) = [], SAMPLE_START
while (y, mth) <= SAMPLE_END:
    expected.append((y, mth)); mth += 1
    if mth > 12: y, mth = y + 1, 1
found = {(int(s[3:7]), int(s[8:10])) for s in CpsSamples}
missing = [f"{y}-{m:02d}" for y, m in expected if (y, m) not in found]
print("Missing months:", missing if missing else "none")

200 samples: 01/2010 to 09/2026
Missing months: ['2025-10']


In [8]:
CpsVariables = [
    "YEAR", 
    "MONTH", 
    "MISH",
    "STATEFIP", 
    "METFIPS", 
    "EARNWEEK2", 
    "AGE", 
    "SEX", 
    "RACE", 
    "HISPAN", 
    "IND", 
    "CLASSWKR",
    "WKSTAT", 
    "EDUC",
]

extract = MicrodataExtract(collection='cps', samples=CpsSamples, variables=CpsVariables)

print('Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Annie Chen, Grace Cooper, Stephanie Richards, Megan Schouweiler, and Michael Westberry. IPUMS CPS: Version 11.0 [dataset]. Minneapolis, MN: IPUMS, 2023. https://doi.org/10.18128/D030.V11.0')

Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Annie Chen, Grace Cooper, Stephanie Richards, Megan Schouweiler, and Michael Westberry. IPUMS CPS: Version 11.0 [dataset]. Minneapolis, MN: IPUMS, 2023. https://doi.org/10.18128/D030.V11.0


In [9]:
if SUBMIT_NEW_EXTRACT:
    extract_id = ipums.submit_extract(extract)
    print("Submitted extract:", extract_id)
    ipums.wait_for_extract(extract)          # can take several minutes
    print("Status:", ipums.extract_status(extract))
    ipums.download_extract(extract_id, download_dir=DATA_DIR)
    print("Extract downloaded to the data folder set in the config cell.")
else:
    print("SUBMIT_NEW_EXTRACT is False: reusing the newest extract already in DATA_DIR.")

SUBMIT_NEW_EXTRACT is False: reusing the newest extract already in DATA_DIR.


In [10]:
# Reference the most recently downloaded files
xml_files = sorted(download_dir.glob("*.xml"), key=lambda f: f.stat().st_mtime)
dat_gz_files = sorted(download_dir.glob("*.dat.gz"), key=lambda f: f.stat().st_mtime)
ddi_file = xml_files[-1]
dat_gz_file = dat_gz_files[-1]

# Read the DDI file, which maps the raw data
with warnings.catch_warnings():
    warnings.simplefilter('ignore')          # IPUMS's citation warning prints a local file path; the citation is printed below
    ddi = readers.read_ipums_ddi(ddi_file)
print(ddi.ipums_citation)

# Get the correct .dat file name from the DDI
dat_file = download_dir / ddi.file_description.filename

if not dat_file.name.endswith('.dat'):
    dat_file = dat_file.with_suffix('.dat')

# Check if the .dat file already exists
if not dat_file.exists():
    try:
        with gzip.open(dat_gz_file, 'rb') as f_in:
            with open(dat_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f"Decompressed data file saved to: {dat_file.name}")
    except EOFError as e:
        print(f"Error decompressing the file: {e}")
        print("The file might be corrupted or incomplete. Please try re-downloading the extract.")
        
# The next step uses the DDI file and the .dat data file to read the IPUMS data into Pandas for later manipulation.

Publications and research reports based on the IPUMS CPS database must cite it appropriately. The citation should include the following:

Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Etienne Breton, Grace Cooper, Julia A. Rivera Drew, Stephanie Richards, David Van Riper, and Kari C.W. Williams. IPUMS CPS: Version 13.0 [dataset]. Minneapolis, MN: IPUMS, 2025. https://doi.org/10.18128/D030.V13.0

The licensing agreement for use of IPUMS CPS data requires that users supply us with the title and full citation for any publications, research reports, or educational materials making use of the data or documentation. Please add your citation to the IPUMS bibliography: http://bibliography.ipums.org/


The dataset is large and must be organized with the DDI file.
The next step breaks the database into small chunks and applies filters to the data to make this task faster.
The first load is slow (about 30 minutes), so the filtered result is cached to disk. To speed this process up, the dataset is pre-filtered as it is loaded. The original dataset has several million observations. The dataset ultimately used for analysis has approximately 600,000 observations (599,085 after these filters, before the minimum wage exclusion in Section 2). Given so many are dropped, below is an explanation for the exclusions:

1. The CPS interviews respondents for four months, takes an eight month break, then interviews them again for four months. The CPS does this in rotation. Therefore, only one eigth of the data in the CPS represents a unique respondent. MISH==8 selects the last month of interviews for a given respondent. This is the last month in which the respondent is asked about their weekly earnings, the primary variable of interest.
2. To allow for a more direct comparison of male and female workers, only full-time workers (35+ hours / week) were considered. These are coded as WKSTAT==11.
3. Similarly, to allow for a more direct comparison of workers only 'Prime Working Age' workers are considered. This is to exclude workers that may have high variation due to school schedules (less than age 25) or may be experiencing wage penalties for approaching retirement (greater than 54).
4. EARNWEEK2 is the endogenous variable of interest. It is what the respondent reported earning from employment before taxes during the week of the 12th in the prior month. Starting in 2023, the CPS began rounding these values to preserve privacy of higher earning respondents. Any respondents making above 150,000 per year were rounded or topcoded to preserve privacy. This removes any variation above this income level and warrants excluding those observations in which EARNWEEK2 is the equivalent of 150,000 per year or more. This likely means all measures of the gender wage gap considered in the analysis are lower than in the population, as men tend to be more represented in higher income and leadership roles. The topcode threshold ($2,884.61 per week) is held fixed over the whole sample and is not adjusted for inflation.
5. Respondents reporting that they make less than the federal minimum wage equivalent (40 hours x 7.25 per hour = 290) were excluded to ensure respondents' answers are consistent with working full time with the more conventional 40 hour work week. Because respondents describe full time has 35+ hours a week, this potentially excludes workers making up to 8.29 per hour but only working 35 hours per week. These workers are unlikely to be affected significantly by EPEWA, so this is considered an acceptable exclusion.

The combination of (4) and (5) above result in the data considering approximately the middle 60% of the U.S. income distribution, with a slight skew to higher earners.

In [11]:
def check_window(df):
    """Stop if the loaded data does not cover exactly the window set in the config cell."""
    ym = df['YEAR'] * 100 + df['MONTH']
    expected = (SAMPLE_START[0] * 100 + SAMPLE_START[1], SAMPLE_END[0] * 100 + SAMPLE_END[1])
    assert (ym.min(), ym.max()) == expected, (
        f"Loaded data covers {ym.min()} to {ym.max()}, not {expected[0]} to {expected[1]}. "
        "The newest file in DATA_DIR is for a different window: set SUBMIT_NEW_EXTRACT = True.")

# Cache of the filtered CPS data. The name includes the sample window, so changing SAMPLE_START/SAMPLE_END
# forces a fresh load instead of silently reusing old data.
RAW_CACHE = DATA_DIR / f"ipums_raw_{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}_{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}.pkl"

if RAW_CACHE.exists():
    ipums_df = pd.read_pickle(RAW_CACHE)
    check_window(ipums_df)
    print(f"Loaded cached data from {RAW_CACHE.name}")
else:
    iter_microdata = readers.read_microdata_chunked(ddi, dat_file, chunksize=3000000)

    # Filters: MISH=8 (unique earnings per person), full-time workers (WKSTAT=11), ages 25-54,
    # weekly earnings between $290 and $2,884.61 (drops topcoded earners and sub-minimum wages)
    ipums_df_list = [
        chunk[
            (chunk["MISH"] == 8) &
            (chunk["WKSTAT"] == 11) &
            (chunk["AGE"] >= 25) &
            (chunk["AGE"] <= 54) &
            (chunk["EARNWEEK2"] < 2884.61) &
            (chunk["EARNWEEK2"] > 290)
        ]
        for chunk in iter_microdata
    ]

    ipums_df = pd.concat(ipums_df_list, ignore_index=True)
    check_window(ipums_df)                    # check BEFORE caching, so a wrong file never poisons the cache
    ipums_df.to_pickle(RAW_CACHE)
    print(f"Saved cache to {RAW_CACHE.name}")

Loaded cached data from ipums_raw_2010-01_2026-09.pkl


In [12]:
# Check the size of the original data extraction
print(len(ipums_df))

599085


In [13]:
# Set display options for future results. This swaps scientific notation for 4 decimal places.

pd.set_option('display.float_format', '{:.4f}'.format)

# Section 2: Minimum Wage Webscrape and Exclusion

The following section builds a table of state minimum wages by year from three sources, then uses it to exclude from the sample workers who reported making less than a state's full time minimum wage weekly earnings in a given state in a given year:

1. The __[historical](https://www.dol.gov/agencies/whd/state/minimum-wage/history)__ US DOL table of state minimum wages, which reports the January 1 rate for each year from 2010 through 2024.
2. The __[current](https://www.dol.gov/agencies/whd/mw-consolidated)__ US DOL consolidated table, a snapshot with a single effective date stated in the table caption (July 1, 2026 as of this run). The code reads the date from the page and uses this table for 2026.
3. **2025, from Wikipedia (provisional).** DOL does not currently publish 2025 rates, so the 2025 column is taken from __[Wikipedia's table of state minimum wages](https://en.wikipedia.org/wiki/List_of_U.S._states_by_minimum_wage)__. This is not an official source. It is flagged with a TODO in the code and should be replaced with official values before any 2025 results are relied upon.

This exclusion is intended to address omitted variable bias of minimum wage increases reducing the gender wage gap. Because women earn less than men, it is likely that women are more likely to work minimum wage than men and benefit more from minimum wage increases than men.

This effect can differ significantly by state. In California, the minimum wage more than doubled between 2010 and 2026 ($8.00 to $16.90). Meanwhile in approximately 20 other states, the federal minimum wage remained constant during this time.

Where the US DOL posts a range of wages for a state minimum, the code assumes the highest of the range as the minimum wages.
Where there is no state minimum wage posted, the code assumes the federal minimum wage.

Once the minimum wage values are collected, they are multiplied by 40 hours to give a weekly threshold. The thresholds are merged with the IPUMS dataframe by state and year, and the exclusion is applied.

**Timing convention:** DOL's historical table reports January 1 rates, so every year through 2024 uses its January 1 rate. 2026 uses the July 1, 2026 snapshot, which may overstate the threshold for January to June in the few jurisdictions that change rates mid-year (for example Alaska, DC and Oregon). The original July 2024 analysis used a snapshot taken in July 2024 for 2024; switching to the January rate changes 26 observations, and the regression coefficients by 0.0001 or less.

**A note on methodology:** This approach only excludes those making less than the minimum wage at 40 hours per week. It is likely that minimum wage increases affect those making slightly above minimum wage by applying wage pressures with those with skill premiums slightly above minimum wage workers but participating in similar work. Sometimes this accounted for by excluding those making more than a certain amount above minimum wage (such as $1 more). In the source data, full-time work is classified as 35 hours per week or more, meaning there could be workers in the existing sample working 114% of minimum wage and working 35 hours per week that would be excluded. This is intended to capture those near the minimum wage, but the original approach of excluding a fixed dollar amount or fixed percentage above the minimum wage may be a superior approach.

In [14]:
# Scrape the DOL historical table of state minimum wages (January 1 rates, one column per year).
url = 'https://www.dol.gov/agencies/whd/state/minimum-wage/history'

response = requests.get(url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
tables = soup.find_all('table', class_='minwage')
assert tables, "No minimum wage tables found. The DOL page layout may have changed."

# The page splits the history into several tables, one per block of years. Read each into a dataframe.
table_frames = []
for table in tables:
    rows = table.find_all('tr')
    headers = [th.get_text() for th in rows[0].find_all('th')]
    data = [dict(zip(headers, [cell.text.strip() for cell in row.find_all(['td', 'th'])])) for row in rows[1:]]
    table_frames.append(pd.DataFrame(data))

# Combine the tables and tidy the columns: rename the state column and keep 2010 onward
concat_mw = pd.concat(table_frames, ignore_index=True)
concat_mw = concat_mw.rename(columns={'State or otherjurisdiction': 'State'})

pre_2010 = [col for col in concat_mw.columns if col != 'State' and col[:4].isdigit() and int(col[:4]) < 2010]
concat_mw = concat_mw.drop(columns=pre_2010)

# One row per jurisdiction (the tables repeat each state; take the first non-missing value in each column)
consol_mw = concat_mw.groupby('State').first().reset_index()

# DOL shows "..." where a state has no minimum of its own: use the federal minimum for that year
federal_row = consol_mw[consol_mw['State'] == 'Federal (FLSA)'].iloc[0]
for year in consol_mw.columns[1:]:
    consol_mw[year] = consol_mw[year].replace('...', federal_row[year])

print(f"Historical minimum wage table: {consol_mw.shape[0]} jurisdictions, years {consol_mw.columns[1]} to {consol_mw.columns[-1]}")

Historical minimum wage table: 55 jurisdictions, years 2010 to 2024


In [15]:
# Data cleaning function

def clean_wage_value(value):

    value_str = str(value)
    
    # Remove notes in parentheses or brackets
    
    if '(' in value_str or '[' in value_str:
        value_str = value_str.split('(')[0].split('[')[0]
    
    # Handle ranges by selecting the higher value
    
    if '-' in value_str:
        value_str = value_str.split('-')[-1]
    
    # Handle slashes by selecting the second (higher) value
    
    if '/' in value_str:
        value_str = value_str.split('/')[-1]
    
    # Format the value as a float, then as a formatted string
    
    try:
        formatted_value = "{:.2f}".format(float(value_str))
    except ValueError:
        # Return original value if conversion fails
        
        formatted_value = value
    
    return formatted_value

# Apply the cleaning function to all year columns

for col in consol_mw.columns[1:]:
    consol_mw[col] = consol_mw[col].apply(clean_wage_value)
    
# Remove territories from data

states_to_remove = ['Puerto Rico', 'U.S. Virgin Islands', 'Guam']
consol_mw = consol_mw[~consol_mw['State'].isin(states_to_remove)]

# Reset the index

consol_mw.reset_index(drop=True, inplace=True)

In [16]:
# Scrape the DOL "consolidated" table of CURRENT state minimum wages.
# This page is a snapshot for one effective date (stated in the table caption), not a fixed year,
# so the year is read from the page instead of being assumed.
url = 'https://www.dol.gov/agencies/whd/mw-consolidated'
response = requests.get(url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', summary="consolidated minimum wage rates.")

# Effective date from the caption, e.g. "(Effective Date: July 1, 2026)"
caption = table.find('caption').get_text(' ', strip=True)
effective = datetime.strptime(re.search(r'Effective Date:\s*([A-Za-z]+ \d{1,2}, \d{4})', caption).group(1), '%B %d, %Y')
fed_wage = float(re.search(r'federal MW of \$(\d+(?:\.\d+)?)', table.get_text(' ', strip=True)).group(1))

# State cells look like "CA $16.90" or "NY $17.00 or $16.00"; where several wages are listed, take the highest.
current_wages = {}
for cell in table.find_all('td'):
    text = cell.get_text(' ', strip=True)
    if '$' not in text:
        continue
    state = text.split('$')[0].strip()
    current_wages[state] = max(float(a) for a in re.findall(r'\$(\d+(?:\.\d+)?)', text))

print(f"DOL snapshot effective {effective:%B %d, %Y}: {len(current_wages)} jurisdictions, federal MW ${fed_wage}")

# Crosswalk acronymns to full length state names

state_acronym_to_full = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    'DC': 'District of Columbia', 'PR': 'Puerto Rico', 'GU': 'Guam', 'VI': 'Virgin Islands', 'CNMI': 'Northern Mariana Islands',
    'AS': 'American Samoa'
}

current_wages_full = {state_acronym_to_full[k]: v for k, v in current_wages.items() if k in state_acronym_to_full}

DOL snapshot effective July 01, 2026: 34 jurisdictions, federal MW $7.25


In [17]:
# =====================================================================================================
# TODO (COME BACK AND FIX): 2025 minimum wages are taken from WIKIPEDIA, which is NOT an official source.
#
# Why: the DOL history table stops at 2024, and the DOL consolidated page is a July 2026 snapshot, so DOL
# currently publishes no 2025 values. This cell is a stopgap so the latest-data run can proceed.
#
# Before relying on any result that includes 2025:
#   1. Replace these values with official ones (DOL's 2025 consolidated table when it is posted, or each
#      state's labor department) and delete this cell.
#   2. Check the states with more than one rate (NJ, NY, OR). The code takes the HIGHEST rate listed,
#      matching the DOL rule used elsewhere, but Wikipedia lists only the standard rate for some states
#      (for example Oregon 2025), so the highest local rate may be missing.
# =====================================================================================================
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_U.S._states_by_minimum_wage'
NEEDS_2025 = 2025 in range(SAMPLE_START[0], SAMPLE_END[0] + 1)

def scrape_wikipedia_wages(year):
    """Return {state name: highest hourly minimum wage listed} from Wikipedia's state minimum wage table."""
    resp = requests.get(WIKI_URL, timeout=30, headers={'User-Agent': 'Mozilla/5.0 (research notebook)'})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    for table in soup.find_all('table', class_='wikitable'):
        header = [c.get_text(' ', strip=True) for c in table.find('tr').find_all(['th', 'td'])]
        cols = [i for i, h in enumerate(header) if h.split()[:1] == [str(year)]]   # e.g. "2025" (ignores footnote marks)
        if header[:1] != ['State'] or not cols:
            continue
        wages = {}
        for row in table.find_all('tr')[1:]:
            cells = [c.get_text(' ', strip=True) for c in row.find_all(['th', 'td'])]
            state = re.sub(r'[\*†\s]+$', '', cells[0]).replace('Washington, D.C.', 'District of Columbia')
            amounts = [float(a) for a in re.findall(r'\$(\d+(?:\.\d+)?)', cells[cols[0]])]
            if amounts:
                wages[state] = max(amounts)
        return wages
    raise ValueError(f"No Wikipedia table with a {year} column found. The page layout may have changed.")

wiki_wages_2025 = scrape_wikipedia_wages(2025) if NEEDS_2025 else {}

if NEEDS_2025:
    print("WARNING: 2025 minimum wages come from Wikipedia (unofficial). See the TODO at the top of this cell.")
    print(f"{len(wiki_wages_2025)} jurisdictions found.")
    print({s: wiki_wages_2025[s] for s in ['California', 'Colorado', 'Washington', 'New Jersey', 'New York', 'Oregon', 'District of Columbia']})
else:
    print("2025 is outside the sample window, so the Wikipedia values are not needed.")

51 jurisdictions found.
{'California': 16.5, 'Colorado': 14.81, 'Washington': 16.66, 'New Jersey': 15.49, 'New York': 16.5, 'Oregon': 14.7, 'District of Columbia': 17.5}


In [18]:
# Assemble ONE minimum wage table covering every year in the study window.
# Sources, in priority order:
#   1. DOL history table (consol_mw): official, one column per year (through 2024 as of this writing)
#   2. DOL consolidated snapshot (current_wages_full): used only for the year of its effective date
#   3. TEMPORARY: Wikipedia values for 2025 (see the TODO cell above; replace with an official source)
needed_years = list(range(SAMPLE_START[0], SAMPLE_END[0] + 1))
year_columns = [str(y) for y in needed_years]

# 1. Make the history table's year columns numeric
for col in [c for c in consol_mw.columns if c.isdigit()]:
    consol_mw[col] = consol_mw[col].replace(r'[\$,]', '', regex=True).astype(float)

# 2. Current snapshot: states not listed on the page use the federal minimum
snap_year = str(effective.year)
if effective.year in needed_years and snap_year not in consol_mw.columns:
    consol_mw[snap_year] = consol_mw['State'].map(current_wages_full).fillna(fed_wage)

# 3. TEMPORARY: 2025 from Wikipedia. States at the federal minimum are listed there too; the Federal row gets fed_wage.
if NEEDS_2025 and '2025' not in consol_mw.columns:
    consol_mw['2025'] = consol_mw['State'].map(wiki_wages_2025).fillna(fed_wage)

# Every year in the window needs a wage for every state, or the run stops here.
missing_years = [y for y in year_columns if y not in consol_mw.columns]
assert not missing_years, f"No minimum wage data for {missing_years}."
assert not consol_mw[year_columns].isna().any().any(), "Some state-years have no minimum wage."

print(consol_mw.loc[consol_mw['State'].isin(['California','Colorado','Washington','Federal (FLSA)']), ['State'] + year_columns[-3:]])

             State    2024    2025    2026
4       California 16.0000 16.5000 16.9000
5         Colorado 14.4200 14.8100 15.1600
9   Federal (FLSA)  7.2500  7.2500  7.2500
48      Washington 16.2800 16.6600 17.1300


In [19]:
# Assertion check to ensure the minimum wages are never declining for a state
# Initialize a list to keep track of states with decreasing wage values

states_with_decreases = []

# Iterate over each row in the DataFrame

for index, row in consol_mw.iterrows():
    
    # Check each year's wage against the previous year's wage
    
    for i in range(1, len(year_columns)):
        previous_year = year_columns[i-1]
        current_year = year_columns[i]
   
        if pd.notna(row[current_year]) and pd.notna(row[previous_year]) and (float(row[current_year]) < float(row[previous_year])):
            states_with_decreases.append((row['State'], current_year))
            break

if states_with_decreases:
    print("States with at least one year where the minimum wage decreased from the previous year:")
    for state, year in states_with_decreases:
        print(f"State: {state}, Year: {year}")
else:
    print("All states have minimum wages equal to or greater than the previous year for every year.")


All states have minimum wages equal to or greater than the previous year for every year.


In [20]:
# Define the statefips mapping dictionary
state_name_to_fips = {
    'Federal (FLSA)': '00',
    'Alabama': '01',
    'Alaska': '02',
    'Arizona': '04',
    'Arkansas': '05',
    'California': '06',
    'Colorado': '08',
    'Connecticut': '09',
    'Delaware': '10',
    'District of Columbia': '11',
    'Florida': '12',
    'Georgia': '13',
    'Hawaii': '15',
    'Idaho': '16',
    'Illinois': '17',
    'Indiana': '18',
    'Iowa': '19',
    'Kansas': '20',
    'Kentucky': '21',
    'Louisiana': '22',
    'Maine': '23',
    'Maryland': '24',
    'Massachusetts': '25',
    'Michigan': '26',
    'Minnesota': '27',
    'Mississippi': '28',
    'Missouri': '29',
    'Montana': '30',
    'Nebraska': '31',
    'Nevada': '32',
    'New Hampshire': '33',
    'New Jersey': '34',
    'New Mexico': '35',
    'New York': '36',
    'North Carolina': '37',
    'North Dakota': '38',
    'Ohio': '39',
    'Oklahoma': '40',
    'Oregon': '41',
    'Pennsylvania': '42',
    'Rhode Island': '44',
    'South Carolina': '45',
    'South Dakota': '46',
    'Tennessee': '47',
    'Texas': '48',
    'Utah': '49',
    'Vermont': '50',
    'Virginia': '51',    
    'Washington': '53',
    'West Virginia': '54',
    'Wisconsin': '55',
    'Wyoming': '56'
}

# Use map function to create a new column 'statefip'

consol_mw['STATEFIP'] = consol_mw['State'].map(state_name_to_fips)

# Check for states that didn't match the mapping (Debugging step)

unmapped_states = consol_mw[consol_mw['STATEFIP'].isna()]['State'].unique()
print("Unmapped States:", unmapped_states)

Unmapped States: []


In [21]:
# Confirm size of IPUMS dataframe before excluding those making less than minimum wage in a given year / state

print(len(ipums_df))

599085


In [22]:
def apply_min_wage_exclusion(df, wages):
    """Drop workers whose weekly earnings are at or below 40 hours x their state's minimum wage for their year.
    `wages` has one row per state (column 'State') and one column per year ('2010', '2011', ...)."""
    long = wages.melt(id_vars='State', var_name='YEAR', value_name='min_wage')
    long['YEAR'] = long['YEAR'].astype(int)
    long['STATEFIP'] = long['State'].map(state_name_to_fips)
    merged = df.merge(long[['STATEFIP', 'YEAR', 'min_wage']], on=['STATEFIP', 'YEAR'], how='left')
    assert merged['min_wage'].notna().all(), "Some workers have no minimum wage for their state and year."
    kept = merged[merged['EARNWEEK2'] > 40 * merged['min_wage']]
    return kept.drop(columns='min_wage').reset_index(drop=True)

In [23]:
# Start from the cached raw data every time, so this cell is safe to rerun
ipums_df = pd.read_pickle(RAW_CACHE)
ipums_df['STATEFIP'] = ipums_df['STATEFIP'].astype(str).str.pad(width=2, fillchar='0')
print("Rows before exclusion:", len(ipums_df))

ipums_df = apply_min_wage_exclusion(ipums_df, consol_mw[['State'] + year_columns])
print("Rows after exclusion:", len(ipums_df))

Rows before exclusion: 599085
Rows after exclusion: 588770


Approximately 10,300 observations (10,315 of 599,085) were excluded for respondents making below the minimum wage for their state and year.

# Section 3: Regression Variables and Controls Preparation

Before proceeding to the regressions, additional variables beyond female need to be generated for other considerations. These variables and their explanations are:

**Comparison Categories**: The primary finding is to evaluate whether EPEWA had an impact on the gender wage gap in Colorado. More specifically, whether the EPEWA requirement to include intended pay ranges in job postings reduced the difference in earnings between men and women. To evaluate this, Colorado is compared to the non pay transparent states. Since EPEWA was passed in 2021, two other states passed similar laws in a time frame that allows for comparisons. California and Washington both had laws come into effect in January 2023. These laws were less stringent than EPEWA, applying only to employers with 15 employees or more where EPEWA applied to all employers. Because EPEWA was in effect for two more years (2021 and 2022) and applies to a wider universe of employers, it is anticipated that the reduction in the gender wage gap for Colorado will be greater than Washington, California, and the remaining 47 states and DC. Because California and Washington passed similar laws with less time and less applicability, they are considered to have a weaker treatment effect and it is anticipated that they will have a reduction in the gender wage gap between Colorado and the 47 states and DC. Therefore for the comparisons, a categorical variable with Colorado, Washington, California, and the remaining 47 states and DC is generated. Note: Hawaii passed a similar pay transparency law that came into effect January 2024. Given this represents such a short time in the sample, Hawaii is included with the non pay transparent states. The City of New York is treated similarly for the law that came into effect in November 2022 - it is hard to isolate the geography of New York City in the data as the CPS groups it with Newark. Since this analysis was first written, the number of states with pay transparency laws has grown to 14. Several of those states are still in the comparison group here, which makes it a less clean control as the sample extends to 2026 and is likely to understate the difference between Colorado and the rest of the country. Hawaii's law took effect in January 2024 and now covers more than two years of the sample, so it is a candidate for separate treatment. An extended analysis that treats the newer states separately is planned.

**Annual Earnings**: The EARNWEEK2 variable is multiplied by 52 to approximately annual earnings for easier comprehension. It is logarithmically transformed for later regression analysis to compare percentage point changes.

**EPEWA**: Defining the period after EPEWA came into effect. There is additional analysis that could be considered that evaluates lags and leads of pay transparency laws coming into effect. As this analysis stands, a simple interpretation of before / after January 2021 when EPEWA came into effect is considered. This is due to other research showing much of the impact in jobs postings including pay ranges occuring in Colorado in the early months of 2021. The post-law period now runs from January 2021 through September 2026, so the EPEWA coefficient also absorbs nominal wage growth over a longer window.

In some of the regression analysis, EPEWA in isolation is used to capture wage inflation effects since January 2021.

**Race Categories**: This is to control by race. When measuring the gender wage gap, there is well established intersectionality of wage penalties possibly due to discrimination. In other words there are three known effects that can explain differences in earnings:

1. Workers tend to be paid less if they are a woman.
2. Workers tend to be paid less if they are a person of color.
3. Workers tend to be paid less if they are a woman of color.

When measuring the gender wage gap (1) adding a control for race ensures the analysis is not measuring (3). This controls for omitted variable bias in Colorado, Washington, or California having different racial make ups than the rest of the United States explaining the difference in gender wage gaps.

For this analysis, hispanic is considered a race. Categorization for __[race](https://cps.ipums.org/cps-action/variables/RACE#codes_section)__ and __[hispanic](https://cps.ipums.org/cps-action/variables/HISPAN#codes_section)__
were sourced from IPUMS.

**Industry**: Another possible explanation for the difference in earnings between men and women is that women tend to enter lower paying industries than men. Whether this behavior is due to discrimination against women in the form of societal conditioning or due to innate preferences of women is out of the scope of this analysis. Therefore this analysis considers both.

In court cases for wage discrimination, where components of EPEWA apply, measurements of damages often rely on within-industry wage comparisons. Therefore this analysis considers industry controls the preferred specification.

The industry categorization comes from __[this IPUMS crosswalk](https://usa.ipums.org/usa/volii/indtoindnaics18.shtml)__ of the CPS 'IND' variable to 2-digit NAICS code industries. The crosswalk has three periods for the years considered (2008-2012, 2013-2017 and 2018-2022). The code finds these columns by pattern, since IPUMS has relabeled them before, and uses the period that contains each worker's survey year. The CPS adopted the 2022 Census industry classification in January 2025, and IPUMS has not published a crosswalk for it. The 34 new codes are therefore assigned to the sector of the nearest lower code in the crosswalk, using IPUMS's own list of 2025-onward codes. The assignments are printed in a table in the code and were checked by hand, but this is a provisional bridge, flagged with a TODO, to be replaced when an official crosswalk exists.

**Age**: Age is controlled for as a proxy for experience. While aging does not necessarily result in experience accumulation, it is impossible to accumulate experience without aging. Controlling for age accounts for possible differences in experience amongst the two male and female sub samples.

**Education**: Education is another determinant of income that may result in omitted variable bias. Men and women have differing rates of educational attaintment. Men have historically had higher levels of education, but this trend has reversed in more recent years as women make a majority of college enrollments. Controlling for education ensures the analysis does not measure these differing rates of education. The educational categorizations come from __[here](https://cps.ipums.org/cps-action/variables/EDUC#codes_section)__.

In [24]:
# Generate state-specific dummy variables for states with pay transparency active as of Jan 2023.

ipums_df['co'] = (ipums_df['STATEFIP'] == '08')
ipums_df['ca'] = (ipums_df['STATEFIP'] == '06')
ipums_df['wa'] = (ipums_df['STATEFIP'] == '53')

# Use CO, WA, and CA variables to create a single comparison state variable

conditions = [
    ipums_df['ca'] == 1,
    ipums_df['co'] == 1,
    ipums_df['wa'] == 1
]
choices = [1, 2, 3]  # 1 for California, 2 for Colorado, 3 for Washington
ipums_df['comp_cat4'] = np.select(conditions, choices, default=0)

# Convert to a categorical variable with labels

comp_cat4_labels = {0: 'Rest of US', 1: 'California', 2: 'Colorado', 3: 'Washington'}
ipums_df['comp_cat4'] = ipums_df['comp_cat4'].map(comp_cat4_labels).astype('category')

# Display the distribution of the categorical variable

display(ipums_df['comp_cat4'].value_counts(dropna=False))

comp_cat4
Rest of US    521535
California     47168
Washington     10122
Colorado        9945
Name: count, dtype: int64

In [25]:
# Generate annual earnings variable. This makes results easier to conceptualize over weekly earnings.

ipums_df['earnyear'] = ipums_df['EARNWEEK2'] * 52

# Generate log annual earnings variable. This will allow for percentage point comparisons for most of the regressions.

ipums_df['log_earnyear'] = np.log(ipums_df['earnyear'])

In [26]:
# Create variable for when Equal Pay for Equal Work Act is in effect

ipums_df['epewa'] = (ipums_df['YEAR'] >= 2021).astype(int)

In [27]:
# Create binary gender variable

ipums_df['female'] = (ipums_df['SEX'] == 2).astype(int)

In [28]:
# Create binary variables for race categories including Hispanic. 
# Hispanic is an ethnicity, not a race, but is useful for grouping by race.

ipums_df['hisp'] = (ipums_df['HISPAN'] >= 100).astype(int)
ipums_df['white'] = ((ipums_df['RACE'] == 100) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['black'] = ((ipums_df['RACE'].isin([200, 801])) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['indian'] = ((ipums_df['RACE'].isin([300, 802])) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['asian'] = ((ipums_df['RACE'].isin([651, 803])) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['other_race'] = ((ipums_df['RACE'].isin([652]) | (ipums_df['RACE'] >= 804)) & (ipums_df['hisp'] == 0)).astype(int)

# Combine the binary flags into a single 'race_cat' variable with Hispanic included

conditions = [
    ipums_df['hisp'] == 1,
    ipums_df['white'] == 1,
    ipums_df['black'] == 1,
    ipums_df['indian'] == 1,
    ipums_df['asian'] == 1,
    ipums_df['other_race'] == 1,
]

# Define the race categories including Hispanic

choices = ['Hispanic', 'White', 'Black', 'Indian', 'Asian', 'Other Race']

# Assign race categories based on conditions

ipums_df['race_cat'] = np.select(conditions, choices, default='Not Specified')

# Print the distribution of the race category variable

race_cat_distribution = ipums_df['race_cat'].value_counts(dropna=False)
display(race_cat_distribution)

assert (ipums_df['race_cat'] != 'Not Specified').all(), "Some workers have an unrecognized RACE/HISPAN code."

race_cat
White         391357
Hispanic       87450
Black          60776
Asian          37176
Indian          7247
Other Race      4764
Name: count, dtype: int64

Note there are no observations that are not specified.

In [29]:
# To create industry categories, load crosswalk from IPUMS

ipums_cw_link = 'https://usa.ipums.org/usa/resources/volii/ind_indnaics_crosswalk_2000_onward_without_code_descriptions.csv'
ipums_cw = pd.read_csv(ipums_cw_link)

In [30]:
def process_ind_codes(code_series):
    if pd.isna(code_series):
        return []
    # Handle float-formatted strings and possible whitespace in entries
    processed_codes = []
    for code in str(code_series).split(','):
        try:
            # Attempt to directly convert to int
            processed_code = int(code.strip())
        except ValueError:
            # If direct conversion fails, treat as float then convert to int
            processed_code = int(float(code.strip()))
        processed_codes.append(processed_code)
    return processed_codes

In [31]:
data_cleaned = ipums_cw.rename(columns={'Unnamed: 0': 'Category'})
data_cleaned['Category'] = data_cleaned['Category'].ffill().fillna('N/A')

# Find the IND CODE columns by pattern, not by exact name. IPUMS relabels periods
# (e.g. "2018 Onward" became "2018-2022"), which broke the hardcoded names.
# Columns look like "2013-2017 ACS/PRCS IND CODE"; the INDNAICS columns do not match.
period_cols = {}   # start year of the period -> column name
for col in data_cleaned.columns:
    m = re.search(r'(\d{4})(?:-\d{4}|\s+Onward)?\s+ACS/PRCS IND CODE\s*$', str(col))
    if m:
        period_cols[int(m.group(1))] = col
period_cols = {y: c for y, c in sorted(period_cols.items()) if y >= 2008}   # CPS samples start in 2010
print("IND code periods found:", period_cols)
assert period_cols, "No ACS/PRCS IND CODE columns found. IPUMS may have changed the crosswalk layout."

# One lookup per period: IND code -> industry category (first category wins, as before)
ind_lookup = {}
for start, col in period_cols.items():
    codes = data_cleaned[col].apply(process_ind_codes)
    lookup = {}
    for category, code_list in zip(data_cleaned['Category'], codes):
        for code in code_list:
            lookup.setdefault(code, category)
    ind_lookup[start] = lookup

IND code periods found: {2008: '2008-2012 ACS/PRCS IND CODE', 2013: '2013-2017 ACS/PRCS IND CODE', 2018: '2018-2022 ACS/PRCS IND CODE'}


In [32]:
# TODO (COME BACK AND FIX): IPUMS's ACS/PRCS crosswalk stops at the 2018-2022 code scheme, but the CPS switched to the
# 2022 Census classification in 2025. This BRIDGES the new 2025+ codes to sectors using the nearest lower code that
# is in the crosswalk (Census industry codes are numbered by sector). Replace it with an official crosswalk for the
# 2022 scheme when IPUMS posts one, and recheck the table below.
resp = requests.get('https://cps.ipums.org/cps/codes/ind_2025_codes.shtml', timeout=30)
resp.raise_for_status()
dl = BeautifulSoup(resp.text, 'html.parser').find('dl', class_='list-fixed-dt')
ind_2025 = {int(dt.get_text(strip=True)): dd.get_text(' ', strip=True) for dt, dd in zip(dl.find_all('dt'), dl.find_all('dd'))}

latest = max(ind_lookup)                         # newest crosswalk period (2018-2022)
known = sorted(ind_lookup[latest])
bridge = []
for code, desc in ind_2025.items():
    if code not in ind_lookup[latest]:
        i = bisect.bisect_right(known, code) - 1
        bridge.append({'IND': code, 'IPUMS description (2025+)': desc,
                       'nearest lower code': known[i] if i >= 0 else None,
                       'assigned category': ind_lookup[latest][known[i]] if i >= 0 else 'N/A'})
bridge_table = pd.DataFrame(bridge)
print(f"{len(ind_2025)} codes in the IPUMS 2025+ list; {len(bridge_table)} not in the crosswalk and bridged:")
display(bridge_table)

# A new "2025" period: everything in the 2018-2022 lookup plus the bridged codes
ind_lookup[2025] = {**ind_lookup[latest], **dict(zip(bridge_table['IND'], bridge_table['assigned category']))}

266 codes in the IPUMS 2025+ list; 34 not in the crosswalk and bridged:


,IND,IPUMS description (2025+),nearest lower code,assigned category
0,480,Not specified type of mining,470,"Mining, Quarrying, and Oil and Gas Extraction:"
1,4681,Other motor vehicle dealers,4680,Retail Trade:
2,4691,"Automotive parts, accessories, and tire retailers",4690,Retail Trade:
3,4771,Furniture and home furnishings retailers,4770,Retail Trade:
4,4796,Electronics and appliance retailers,4795,Retail Trade:
5,4871,"Building material and supplies dealers, except...",4870,Retail Trade:
6,4881,Hardware retailers,4880,Retail Trade:
7,4891,Lawn and garden equipment and supplies retailers,4890,Retail Trade:
8,4973,Convenience retailers and vending machine oper...,4972,Retail Trade:
9,4981,Specialty food retailers,4980,Retail Trade:


In [33]:
def lookup_category(year, ind_code):
    """Industry category for an IND code: use the period that contains the year, then fall back to the
    nearest other periods (ties go to the later period), as the original did."""
    primary = max((s for s in ind_lookup if s <= year), default=min(ind_lookup))
    order = [primary] + sorted((s for s in ind_lookup if s != primary), key=lambda s: (abs(s - primary), -s))
    for s in order:
        cat = ind_lookup[s].get(ind_code)
        if cat is not None:
            return cat
    return 'N/A'

ipums_df['ind_cat20'] = [lookup_category(y, c) for y, c in zip(ipums_df['YEAR'], ipums_df['IND'])]
ipums_df['ind_cat20_label'] = ipums_df['ind_cat20']

print("Count of 'N/A' in ind_cat20_label:", (ipums_df['ind_cat20_label'] == 'N/A').sum())

Count of 'N/A' in ind_cat20_label: 0


In [34]:
# Coverage check: unmatched workers by year. Watch the newest years, because the crosswalk
# only goes through 2018-2022 and newer IND codes may be unmapped.
unmatched = (ipums_df['ind_cat20'] == 'N/A').groupby(ipums_df['YEAR']).agg(['sum', 'mean'])
unmatched.columns = ['unmatched_rows', 'share']
print(unmatched.round(4))
assert unmatched['share'].max() < 0.01, "More than 1% of workers in some year have no industry match."

      unmatched_rows  share
YEAR                       
2010               0 0.0000
2011               0 0.0000
2012               0 0.0000
2013               0 0.0000
2014               0 0.0000
2015               0 0.0000
2016               0 0.0000
2017               0 0.0000
2018               0 0.0000
2019               0 0.0000
2020               0 0.0000
2021               0 0.0000
2022               0 0.0000
2023               0 0.0000
2024               0 0.0000
2025               0 0.0000
2026               0 0.0000


Note there are no observations in the 'N/A' category, meaning every observation was assigned.

In [35]:
# Convert AGE to a categorical variable named 'age'

ipums_df['age'] = ipums_df['AGE'].astype('category')

# Optionally, to name categories based on the integer value of AGE:

ipums_df['age'] = ipums_df['age'].cat.rename_categories(lambda x: f"Age {x}")

# Check the first few rows to confirm the conversion

age_distribution = ipums_df['age'].value_counts(dropna=False)
age_distribution = age_distribution.sort_index()
print(age_distribution)

age
Age 25    17805
Age 26    18796
Age 27    19518
Age 28    19651
Age 29    20350
Age 30    20672
Age 31    20975
Age 32    20574
Age 33    20522
Age 34    20019
Age 35    20087
Age 36    19887
Age 37    19679
Age 38    19427
Age 39    19434
Age 40    19669
Age 41    19769
Age 42    19305
Age 43    19222
Age 44    19085
Age 45    19269
Age 46    19408
Age 47    19266
Age 48    19506
Age 49    19394
Age 50    19597
Age 51    19971
Age 52    19396
Age 53    19434
Age 54    19083
Name: count, dtype: int64


In [36]:
# EDUC crosswalk

educ_crosswalk = {
    2: "None or preschool",
    10: "Grades 1, 2, 3, or 4",
    20: "Grades 5 or 6",
    30: "Grades 7 or 8",
    40: "Grade 9",
    50: "Grade 10",
    60: "Grade 11",
    71: "12th grade, no diploma",
    73: "High school diploma or equivalent",
    81: "Some college but no degree",
    91: "Associate's degree, occupational/vocational",
    92: "Associate's degree, academic program",
    111: "Bachelor's degree",
    123: "Master's degree",
    124: "Professional school degree",
    125: "Doctorate degree",
}


# Convert EDUC to a categorical variable named 'educ'

ipums_df['educ'] = ipums_df['EDUC'].map(educ_crosswalk).astype('category')

# Check the conversion by displaying the first few entries

education_distribution = ipums_df['educ'].value_counts(dropna=False)
education_distribution = education_distribution.sort_index()
print(education_distribution)

assert ipums_df['educ'].notna().all(), "Some workers have an EDUC code missing from educ_crosswalk."

educ
12th grade, no diploma                           4908
Associate's degree, academic program            39999
Associate's degree, occupational/vocational     30269
Bachelor's degree                              157657
Doctorate degree                                11520
Grade 10                                         4944
Grade 11                                         6648
Grade 9                                          5008
Grades 1, 2, 3, or 4                             1766
Grades 5 or 6                                    4775
Grades 7 or 8                                    4118
High school diploma or equivalent              150750
Master's degree                                 66119
None or preschool                                 717
Professional school degree                       7987
Some college but no degree                      91585
Name: count, dtype: int64


# Section 4: Primary Regression Analysis

The following section runs the primary regressions and produces the comparison charts.

The primary model uses a regression of the logarithmically transformed annual earnings as the endogenous (y) variable. It uses gender, the EPEWA time period, age, (sometimes) industry, race, and education as exogenous (x) variables.

It also uses the interaction term, female:epewa. This is the primary finding of the regressions. This is the reduction in the gender wage gap attributable to EPEWA and the other pay transparency laws using this difference in difference experimental design.

The code in this section fits the model separately for each comparison group (Colorado, Washington, California, and the rest of the country, as defined in Section 3), first without industry controls and then with them. All regressions use robust (HC3) standard errors. Full regression summaries for any group can be printed from the stored models, for example `models_ind['Colorado'].summary()`.

In [37]:
models_no_ind, results_df = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND)
print('Regression Results for Key Variables by comp_cat4 Category, No Industry Controls')
display(results_df.round(4))

Regression Results for Key Variables by comp_cat4 Category, No Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,pre_epewa_wage_gap,post_epewa_wage_gap
California,-0.1493,0.0000,0.0266,0.0009,0.8507,0.8772
Colorado,-0.2180,0.0000,0.0897,0.0000,0.7820,0.8717
Rest of US,-0.2146,0.0000,0.0180,0.0000,0.7854,0.8034
Washington,-0.2207,0.0000,0.0615,0.0003,0.7793,0.8408


In [38]:
fig = plot_group_gaps(results_df, "Pre- and Post-EPEWA Wage Gaps by Comparison Group", "Age, Education, and Race",
                      save_as="groups_no_industry")

The interaction coefficients are significant at the 5% level for all four comparison groups (the largest p-value is 0.0009, for California). Without industry controls, the change in women's earnings as a percentage of men's is 9.0 percentage points in Colorado, 6.2 in Washington, 2.7 in California and 1.8 in the rest of the country.

Women's earnings in Colorado rose from 78.2 to 87.2 cents on the dollar of men's earnings after EPEWA came into effect, a larger change than in Washington (77.9 to 84.1), California (85.1 to 87.7) or the rest of the country (78.5 to 80.3). The relative sizes are consistent with the anticipated treatment effects.

In [39]:
models_ind, results_df = run_by_group(ipums_df, 'comp_cat4', FORMULA_IND)
print('Regression Results for Key Variables by comp_cat4 Category, Industry Controls')
display(results_df.round(4))

Regression Results for Key Variables by comp_cat4 Category, Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,pre_epewa_wage_gap,post_epewa_wage_gap
California,-0.1366,0.0000,0.0266,0.0008,0.8634,0.8900
Colorado,-0.1778,0.0000,0.0834,0.0000,0.8222,0.9056
Rest of US,-0.1815,0.0000,0.0179,0.0000,0.8185,0.8365
Washington,-0.1779,0.0000,0.0622,0.0002,0.8221,0.8843


In [40]:
fig = plot_group_gaps(results_df, "Pre- and Post-EPEWA Wage Gaps by Comparison Group", "Age, Education, Race, and Industry",
                      save_as="groups_industry")

With industry controls, the change is 8.3 percentage points in Colorado, 6.2 in Washington, 2.7 in California and 1.8 in the rest of the country. All four are significant at the 5% level (the largest p-value is 0.0008).

With industry controls, women's earnings in Colorado rose from 82.2 to 90.6 cents on the dollar of men's earnings, compared with 82.2 to 88.4 in Washington, 86.3 to 89.0 in California and 81.9 to 83.7 in the rest of the country. The relative values remain consistent with the anticipated treatment effects.

# Section 5: State Ranking and Charts

This section of code adds names to the dataset for each individual state (this code could be moved to section 3 but wasn't needed until this point).

To evaluate how the states rank against each other and the national average for the gender wage gap, the code then runs the regression model for each state and compiles the results into a dataframe. In this dataframe, the code quantifies the measurements of the gender wage gap into the typical 'cents on the dollar' interpretation (women's earnings as a percentage of men's). To evaluate the effects of EPEWA, the code adds the change to the pre-trend value to approximate an after EPEWA value.

An asterisk next to a state on a chart means the relevant coefficient is **not** statistically significant at the 10% level (p > 0.10): the pre-law gap for the first chart, and the change in the gap for the other two. The asterisks are applied by the code, not by hand. Across the 50 states and DC, 22 show a statistically significant change at the 10% level without industry controls and 23 with them (20 and 22 in the July 2024 analysis). At the 5% level the counts are 20 and 20 (14 and 16 previously).

This process is repeated with and without industry controls.

In [41]:
# Give state names to the STATEFIP codes (inverse of the name -> FIPS mapping built in Section 2)
fips_to_state = {fips: name for name, fips in state_name_to_fips.items() if name != 'Federal (FLSA)'}

ipums_df['STATEFIP'] = ipums_df['STATEFIP'].astype(str).str.pad(2, fillchar='0')
ipums_df['state'] = ipums_df['STATEFIP'].map(fips_to_state).astype('category')

assert ipums_df['state'].notna().all(), "Some STATEFIP codes are not in the state mapping."
display(ipums_df[['STATEFIP', 'state']].head())

,STATEFIP,state
0,01,Alabama
1,01,Alabama
2,01,Alabama
3,09,Connecticut
4,09,Connecticut


In [42]:
state_models_no_ind, results_df = run_by_group(ipums_df, 'state', FORMULA_NO_IND, include_national=True)
print('Individual State Regression Results for Key Variables, No Industry Controls')
display(results_df.round(4))

Individual State Regression Results for Key Variables, No Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,pre_epewa_wage_gap,post_epewa_wage_gap
Alabama,-0.2507,0.0000,0.0302,0.0803,0.7493,0.7795
Alaska,-0.2292,0.0000,0.0607,0.0053,0.7708,0.8315
Arizona,-0.1683,0.0000,-0.0100,0.5946,0.8317,0.8217
Arkansas,-0.2137,0.0000,0.0516,0.0046,0.7863,0.8380
California,-0.1493,0.0000,0.0266,0.0009,0.8507,0.8772
Colorado,-0.2180,0.0000,0.0897,0.0000,0.7820,0.8717
Connecticut,-0.1922,0.0000,0.0251,0.2605,0.8078,0.8329
Delaware,-0.1721,0.0000,0.0157,0.4872,0.8279,0.8437
District of Columbia,-0.0686,0.0000,0.0032,0.8365,0.9314,0.9346
Florida,-0.1713,0.0000,0.0247,0.0279,0.8287,0.8534


In [43]:
ranking_no_ind = results_df.rename(columns={'pre_epewa_wage_gap': 'Pre_EPEWA_Gap', 'post_epewa_wage_gap': 'Post_EPEWA_Gap'})
display(ranking_no_ind.round(4))

,female_coef,female_pval,interaction_coef,interaction_pval,Pre_EPEWA_Gap,Post_EPEWA_Gap
Alabama,-0.2507,0.0000,0.0302,0.0803,0.7493,0.7795
Alaska,-0.2292,0.0000,0.0607,0.0053,0.7708,0.8315
Arizona,-0.1683,0.0000,-0.0100,0.5946,0.8317,0.8217
Arkansas,-0.2137,0.0000,0.0516,0.0046,0.7863,0.8380
California,-0.1493,0.0000,0.0266,0.0009,0.8507,0.8772
Colorado,-0.2180,0.0000,0.0897,0.0000,0.7820,0.8717
Connecticut,-0.1922,0.0000,0.0251,0.2605,0.8078,0.8329
Delaware,-0.1721,0.0000,0.0157,0.4872,0.8279,0.8437
District of Columbia,-0.0686,0.0000,0.0032,0.8365,0.9314,0.9346
Florida,-0.1713,0.0000,0.0247,0.0279,0.8287,0.8534


In [44]:
fig = plot_state_bars(ranking_no_ind, 'Pre_EPEWA_Gap', 'female_pval', "Women's Earnings As % of Men's",
                      "Age, Education, and Race", PRE_PERIOD, [0.7, 1], save_as="states_pre_gap_no_industry")

In [45]:
fig = plot_state_bars(ranking_no_ind, 'interaction_coef', 'interaction_pval', "Change in Women's Earnings As % of Men's",
                      "Age, Education, and Race", f"{PRE_PERIOD} to {POST_PERIOD}", [-0.05, 0.1], save_as="states_change_no_industry")

In [46]:
fig = plot_state_bars(ranking_no_ind, 'Post_EPEWA_Gap', 'interaction_pval', "Women's Earnings As % of Men's",
                      "Age, Education, and Race", POST_PERIOD, [0.7, 1], save_as="states_post_gap_no_industry")

In [47]:
state_models_ind, results_df = run_by_group(ipums_df, 'state', FORMULA_IND, include_national=True)
print('Individual State Regression Results for Key Variables, Industry Controls')
display(results_df.round(4))

Individual State Regression Results for Key Variables, Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,pre_epewa_wage_gap,post_epewa_wage_gap
Alabama,-0.2192,0.0000,0.0286,0.0955,0.7808,0.8093
Alaska,-0.1893,0.0000,0.0632,0.0030,0.8107,0.8739
Arizona,-0.1476,0.0000,-0.0068,0.7137,0.8524,0.8456
Arkansas,-0.1879,0.0000,0.0498,0.0058,0.8121,0.8620
California,-0.1366,0.0000,0.0266,0.0008,0.8634,0.8900
Colorado,-0.1778,0.0000,0.0834,0.0000,0.8222,0.9056
Connecticut,-0.1765,0.0000,0.0259,0.2384,0.8235,0.8493
Delaware,-0.1601,0.0000,0.0221,0.3214,0.8399,0.8620
District of Columbia,-0.0568,0.0000,0.0056,0.7121,0.9432,0.9487
Florida,-0.1655,0.0000,0.0266,0.0164,0.8345,0.8611


In [48]:
ranking_ind = results_df.rename(columns={'pre_epewa_wage_gap': 'Pre_EPEWA_Gap', 'post_epewa_wage_gap': 'Post_EPEWA_Gap'})
display(ranking_ind.round(4))

,female_coef,female_pval,interaction_coef,interaction_pval,Pre_EPEWA_Gap,Post_EPEWA_Gap
Alabama,-0.2192,0.0000,0.0286,0.0955,0.7808,0.8093
Alaska,-0.1893,0.0000,0.0632,0.0030,0.8107,0.8739
Arizona,-0.1476,0.0000,-0.0068,0.7137,0.8524,0.8456
Arkansas,-0.1879,0.0000,0.0498,0.0058,0.8121,0.8620
California,-0.1366,0.0000,0.0266,0.0008,0.8634,0.8900
Colorado,-0.1778,0.0000,0.0834,0.0000,0.8222,0.9056
Connecticut,-0.1765,0.0000,0.0259,0.2384,0.8235,0.8493
Delaware,-0.1601,0.0000,0.0221,0.3214,0.8399,0.8620
District of Columbia,-0.0568,0.0000,0.0056,0.7121,0.9432,0.9487
Florida,-0.1655,0.0000,0.0266,0.0164,0.8345,0.8611


In [49]:
fig = plot_state_bars(ranking_ind, 'Pre_EPEWA_Gap', 'female_pval', "Women's Earnings As % of Men's",
                      "Age, Education, Race, and Industry", PRE_PERIOD, [0.7, 1], save_as="states_pre_gap_industry")

In [50]:
fig = plot_state_bars(ranking_ind, 'interaction_coef', 'interaction_pval', "Change in Women's Earnings As % of Men's",
                      "Age, Education, Race, and Industry", f"{PRE_PERIOD} to {POST_PERIOD}", [-0.04, 0.085], save_as="states_change_industry")

In [51]:
fig = plot_state_bars(ranking_ind, 'Post_EPEWA_Gap', 'interaction_pval', "Women's Earnings As % of Men's",
                      "Age, Education, Race, and Industry", POST_PERIOD, [0.7, 1], save_as="states_post_gap_industry")

# Section 6: Supplementary Analysis and Robustness Checks

This section of code seeks to analyze some supplementary considerations. These include:

1. Is the reduction in the gender wage gap in Colorado being driven by women's wages going up, or men's wages going down? This is accomplished by repeating the primary regression but with men only in the sample. This compares Colorado men's wages to the rest of the men in the United States.
2. Is the reduction within Colorado determined by a specific industry? This was evaluated by splitting the Colorado subsample into industry-specific samples and running the regression without industry controls. The analysis runs into significant issues due to small sample size. An industry is reported only if each of its four gender-by-period groups has at least 10 observations and the regression returns valid standard errors; the rest are listed as not reported.
3. Is the reduction in the gender wage gap in Colorado driven by urban or rural parts of the state? Urban Colorado is defined as the Denver, Colorado Springs, Fort Collins, Greeley, Pueblo and Boulder metropolitan areas (from the IPUMS METFIPS variable). Again, this runs into significant issues due to small sample size.
4. What is the increase in women's wages relative to men's in terms of dollars (not percentage points)? These are nominal dollars as reported in each year; no inflation adjustment is made.

In [52]:
FORMULA_MEN = 'log_earnyear ~ C(epewa) + C(co) + C(epewa):C(co) + C(age) + C(ind_cat20_label) + C(race_cat) + C(educ)'

male_subset = ipums_df[ipums_df['female'] == 0]
model_men = smf.ols(FORMULA_MEN, data=male_subset).fit(cov_type='HC3')

print('Regression Measuring Change in Colorado Male Wages Relative to US, Before and After EPEWA')
print('Observations:', int(model_men.nobs))
display(term_table(model_men, ['epewa', 'C(co)']))

Regression Measuring Change in Colorado Male Wages Relative to US, Before and After EPEWA
Observations: 322910


,coef,std_err,p_value
C(epewa)[T.1],0.2340,0.0016,0.0000
C(co)[T.True],0.0395,0.0064,0.0000
C(epewa)[T.1]:C(co)[T.True],0.0254,0.0131,0.0530


There is no evidence of a drop in Colorado men's wages relative to men in the rest of the United States. Colorado men's wages rose by 2.5 percentage points more than men's elsewhere after EPEWA (p = 0.053), which is significant at the 10% level but not at the 5% level. In the July 2024 analysis the figure was 1.6 points (p = 0.32). Some of the narrowing of the gap in Colorado may therefore reflect broader wage growth in the state.

In [53]:
FORMULA_CO_IND = 'log_earnyear ~ C(epewa) + C(female) + C(female):C(epewa) + C(age) + C(race_cat) + C(educ)'

co_df = ipums_df[ipums_df['co'] == 1]
rows = []
for industry, subset in co_df.groupby('ind_cat20_label', observed=True):
    cells = subset.groupby(['female', 'epewa']).size()      # observations in each gender x period cell
    row = {'Industry Label': industry, 'n': len(subset), 'min_cell': cells.min() if len(cells) == 4 else 0}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')   # degenerate fits are screened out below (group sizes, p-values)
            model, key = fit_did(subset, FORMULA_CO_IND)
            row.update({'max_leverage': model.get_influence().hat_matrix_diag.max(), **key})
    except Exception as e:                                   # e.g. a singular design matrix
        row.update({'max_leverage': np.nan, 'error': str(e)[:60]})
    rows.append(row)

co_industry_diag = pd.DataFrame(rows).set_index('Industry Label').sort_values('max_leverage', ascending=False)
display(co_industry_diag.round(4))

,n,min_cell,max_leverage,female_coef,female_pval,interaction_coef,interaction_pval,error
Industry Label,,,,,,,,
"Arts, Entertainment, and Recreation:",173,16,1.0000,-0.1201,1.0000,0.1343,1.0000,NaN
Real Estate and Rental and Leasing:,211,24,1.0000,-0.0879,0.2932,0.1080,0.5531,NaN
Administrative and Support and Waste Management and Remediation Services:,357,24,1.0000,-0.0949,0.1301,-0.0711,0.6031,NaN
Public Administration:,608,63,1.0000,-0.1419,0.0002,-0.1006,0.1814,NaN
"Agriculture, Forestry, Fishing and Hunting:",107,10,1.0000,-0.4448,0.0236,0.1801,0.5813,NaN
Other Services (Except Public Administration):,374,29,1.0000,-0.2566,0.0000,0.1447,0.2273,NaN
Health Care and Social Assistance:,1258,67,1.0000,-0.1280,0.0001,0.0683,0.2940,NaN
"Mining, Quarrying, and Oil and Gas Extraction:",140,6,1.0000,-0.3003,0.0816,0.1185,0.6743,NaN
Wholesale Trade:,267,14,1.0000,-0.1608,0.0289,-0.1362,0.4565,NaN


In [54]:
MIN_CELL = 10   # minimum observations in each gender x period group for an industry to be reported

# A fit is unusable if the HC3 standard errors degenerate: missing or exactly-1.0 p-values
pvals = co_industry_diag[['female_pval', 'interaction_pval']]
enough_obs = co_industry_diag['min_cell'] >= MIN_CELL
valid_fit = pvals.notna().all(axis=1) & (pvals < 0.9999).all(axis=1)
usable = enough_obs & valid_fit

print(f"Reported industries: {usable.sum()} of {len(co_industry_diag)}")
print("Not reported (fewer than %d in a gender x period group):" % MIN_CELL, ", ".join(co_industry_diag.index[~enough_obs]) or "none")
print("Not reported (degenerate fit):", ", ".join(co_industry_diag.index[enough_obs & ~valid_fit]) or "none")

co_industry_results = co_industry_diag[usable].copy()
co_industry_results['Pre-EPEWA Wage Gap'] = 1 + co_industry_results['female_coef']
co_industry_results['Post-EPEWA Wage Gap'] = co_industry_results['Pre-EPEWA Wage Gap'] + co_industry_results['interaction_coef']

# Asterisk where the interaction p-value is above 0.10
co_industry_results.index = [f"{name}*" if p > 0.10 else name
                             for name, p in zip(co_industry_results.index, co_industry_results['interaction_pval'])]
co_industry_results = co_industry_results.sort_values('interaction_pval')

display(co_industry_results[['n', 'min_cell', 'female_coef', 'female_pval', 'interaction_coef', 'interaction_pval',
                             'Pre-EPEWA Wage Gap', 'Post-EPEWA Wage Gap']].round(4))

Reported industries: 15 of 20
Not reported (fewer than 10 in a gender x period group): Mining, Quarrying, and Oil and Gas Extraction:, Utilities:, Management of Companies and Enterprises:
Not reported (degenerate fit): Arts, Entertainment, and Recreation:, Transportation and Warehousing:


,n,min_cell,female_coef,female_pval,interaction_coef,interaction_pval,Pre-EPEWA Wage Gap,Post-EPEWA Wage Gap
"Professional, Scientific, and Technical Services:",1013,115,-0.3010,0.0000,0.2701,0.0000,0.6990,0.9691
Manufacturing:,826,39,-0.2136,0.0000,0.1925,0.0216,0.7864,0.9789
Retail Trade:,874,79,-0.1876,0.0000,0.1520,0.0316,0.8124,0.9645
Educational Services:*,991,63,-0.1165,0.0007,0.1003,0.1486,0.8835,0.9838
Public Administration:*,608,63,-0.1419,0.0002,-0.1006,0.1814,0.8581,0.7576
Other Services (Except Public Administration):*,374,29,-0.2566,0.0000,0.1447,0.2273,0.7434,0.8881
Health Care and Social Assistance:*,1258,67,-0.1280,0.0001,0.0683,0.2940,0.8720,0.9404
Wholesale Trade:*,267,14,-0.1608,0.0289,-0.1362,0.4565,0.8392,0.7030
Accommodation and Food Services:*,484,26,-0.0670,0.1691,0.0771,0.5186,0.9330,1.0101
Real Estate and Rental and Leasing:*,211,24,-0.0879,0.2932,0.1080,0.5531,0.9121,1.0201


Three industries show a statistically significant increase in women's relative earnings at the 5% level: Professional, Scientific, and Technical Services (+27.0 percentage points, with women's earnings rising from 69.9 to 96.9 cents on the dollar), Manufacturing (+19.3 points, 78.6 to 97.9 cents, p = 0.022) and Retail Trade (+15.2 points, 81.2 to 96.5 cents, p = 0.032). Manufacturing was not significant in the July 2024 analysis. Mining, Utilities and Management of Companies are not reported because a gender-by-period group has fewer than 10 observations, and Arts, Entertainment, and Recreation and Transportation and Warehousing are not reported because their regressions return degenerate standard errors.
These three industries seem to be driving the change in the gender wage gap, though other industries may also be contributing. It is difficult to tell to what magnitude other industries are contributing.

Which industries are reported depends partly on numerical details of the robust standard errors in very small samples, so these industry results should be read as indicative.

In [55]:
# According to the Colorado demographer, the 'Front Range' urban corridor ranges from 70-90% of the state
# https://storymaps.arcgis.com/stories/f622f3746d7649749b103e1e2b018a35
# Metropolitan areas used as "urban" Colorado (METFIPS codes, 2014 onward):
# https://cps.ipums.org/cps/codes/metfips_2014onward_codes.shtml
URBAN_CO_METROS = {19740: 'Denver-Aurora-Lakewood', 
                   17820: 'Colorado Springs', 
                   22660: 'Fort Collins',
                   24540: 'Greeley', 
                   39380: 'Pueblo', 
                   14500: 'Boulder'}

ipums_df['urban_co'] = ((ipums_df['METFIPS'].isin(URBAN_CO_METROS)) & (ipums_df['co'] == 1)).astype(int)
ipums_df['rural_co'] = ((ipums_df['urban_co'] == 0) & (ipums_df['co'] == 1)).astype(int)

In [56]:
# Because the samples are so small, controls were dropped to take the simplest measure possible
FORMULA_SIMPLE = 'log_earnyear ~ C(epewa) + C(female) + C(epewa):C(female)'

for label, flag in [('Urban', 'urban_co'), ('Rural', 'rural_co')]:
    subset = ipums_df[ipums_df[flag] == 1]
    model = smf.ols(FORMULA_SIMPLE, data=subset).fit(cov_type='HC3')
    print(f'Simple Gender Wage Gap Model Results for {label} Colorado, No Controls (observations: {int(model.nobs):,})')
    display(term_table(model, ['epewa', 'female']))

Simple Gender Wage Gap Model Results for Urban Colorado, No Controls (observations: 8,713)


,coef,std_err,p_value
C(epewa)[T.1],0.2826,0.0160,0.0000
C(female)[T.1],-0.1609,0.0122,0.0000
C(epewa)[T.1]:C(female)[T.1],0.1221,0.0231,0.0000


Simple Gender Wage Gap Model Results for Rural Colorado, No Controls (observations: 1,232)


,coef,std_err,p_value
C(epewa)[T.1],0.2476,0.0394,0.0000
C(female)[T.1],-0.2287,0.0313,0.0000
C(epewa)[T.1]:C(female)[T.1],0.1108,0.0585,0.0581


The interaction coefficient is 12.2 percentage points for urban Colorado (p < 0.001) and 11.1 for rural Colorado (p = 0.058, significant at the 10% level but not the 5% level). The magnitudes are similar, and there does not appear to be a large difference between the rural and urban reduction in the gender wage gap using the simplest measure.

In [57]:
# Dollar value of the primary finding, in nominal dollars per year: industry controls
FORMULA_IND_DOLLARS = FORMULA_IND.replace('log_earnyear', 'earnyear')
models_dollars_ind, dollars_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_IND_DOLLARS, add_gaps=False)

print('Dollar value of the change in the gender gap, industry controls ($ per year)')
display(dollars_ind.round({'female_coef': 0, 'interaction_coef': 0, 'female_pval': 4, 'interaction_pval': 4}))

Dollar value of the change in the gender gap, industry controls ($ per year)


,female_coef,female_pval,interaction_coef,interaction_pval
California,-8116.0000,0.0000,1069.0000,0.0472
Colorado,-10185.0000,0.0000,3935.0000,0.0027
Rest of US,-9496.0000,0.0000,-472.0000,0.0026
Washington,-10306.0000,0.0000,1885.0000,0.1104


The reduction in the gender wage gap with industry controls in Colorado is approximately $3,935 per year (p = 0.003), in nominal dollars as reported in each year.

In [58]:
# Dollar value of the primary finding, in nominal dollars per year: no industry controls
FORMULA_NO_IND_DOLLARS = FORMULA_NO_IND.replace('log_earnyear', 'earnyear')
models_dollars_no_ind, dollars_no_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND_DOLLARS, add_gaps=False)

print('Dollar value of the change in the gender gap, no industry controls ($ per year)')
display(dollars_no_ind.round({'female_coef': 0, 'interaction_coef': 0, 'female_pval': 4, 'interaction_pval': 4}))

Dollar value of the change in the gender gap, no industry controls ($ per year)


,female_coef,female_pval,interaction_coef,interaction_pval
California,-8887.0000,0.0000,1101.0000,0.0438
Colorado,-12612.0000,0.0000,4264.0000,0.0015
Rest of US,-11291.0000,0.0000,-461.0000,0.0038
Washington,-12907.0000,0.0000,1820.0000,0.1294


When removing industry controls, the reduction grows to approximately $4,264 per year (p = 0.002). In nominal dollars the gap in the rest of the country widened by about $460 over the same period, mostly reflecting nominal wage growth, so Colorado's figure is measured against a rising baseline.

The following cell saves results.

In [59]:
# ---- Save the key results for this sample window, so runs can be compared later --------------------------
RESULTS_DIR = DATA_DIR / 'results' / f"{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}_{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_, group_no_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND)
_, group_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_IND)

tables = {
    'groups_no_industry': group_no_ind,
    'groups_industry': group_ind,
    'states_no_industry': ranking_no_ind,
    'states_industry': ranking_ind,
    'colorado_dollars_no_industry': dollars_no_ind,
    'colorado_dollars_industry': dollars_ind,
    'colorado_by_industry': co_industry_results,
    'colorado_men_only': term_table(model_men, ['epewa', 'C(co)']),
}
for name, table in tables.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv")

run_info = pd.Series({
    'sample_start': f"{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}",
    'sample_end': f"{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}",
    'monthly_samples': len(CpsSamples),
    'rows_after_exclusion': len(ipums_df),
    'run_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
})
run_info.to_csv(RESULTS_DIR / 'run_info.csv', header=False)
print(f"Saved {len(tables)} tables to {RESULTS_DIR.relative_to(DATA_DIR)}")
print(run_info)

Saved 8 tables to results\2010-01_2026-09
sample_start                     2010-01
sample_end                       2026-09
monthly_samples                      200
rows_after_exclusion              588770
run_date                2026-10-15 17:11
dtype: object


# Conclusion

The key findings from this analysis, using monthly CPS data from January 2010 through September 2026, are:

> Colorado has reduced the gender wage gap faster than any other state since implementing EPEWA. Washington shows a similar but smaller result, and California a smaller one.
> The reduction in Colorado is equivalent to women being paid approximately $3,935 more per year with industry controls and $4,264 more without, measured in nominal dollars as reported in each year.

This was determined using a difference in difference experimental design. These results were consistent when controlling for industry and not. This analysis utilizes the middle 60% of the U.S. income distribution. The relevant amounts are 8-9 cents on the dollar for Colorado (9.0 without industry controls and 8.3 with), 6 cents for Washington, about 3 cents for California (2.7), and about 2 cents for the rest of the country (1.8).

The dollar figures are not adjusted for inflation, so they should be read as a lower bound in present-value terms. Part of the increase over the July 2024 figures ($3,304 and $3,358) reflects price growth, because the nominal dollar gap in the rest of the country also widened over the longer window. The percentage-point results are less affected.

The reduction in the gender wage gap in Colorado is larger when the industry control is not applied. This implies women are transitioning to higher paying industries after EPEWA came into effect.

> Colorado went from a near-median gender wage gap among the 50 states and DC (21st of 51 before the law) to ranking 2nd after EPEWA when controlling for industry, behind only DC, and 3rd without industry controls, behind DC and California. It had the largest reduction in the gender wage gap among the 50 states and DC in both specifications.

At the state level, 20 of 51 jurisdictions show a statistically significant change at the 5% level in both specifications (up from 14 and 16 in the July 2024 analysis). Not all state-level findings are significant, but their cardinality can provide some insight.

> The gender wage gap in Colorado is closing mainly because women's wages are rising relative to men's, not because men's wages are falling.

There is no evidence that Colorado men's wages are falling behind the nation. Colorado men's wages rose 2.5 percentage points more than men's elsewhere after EPEWA (p = 0.053), so some of the narrowing may reflect broader wage growth in Colorado.

> The reduction in Colorado is concentrated in Professional, Scientific, and Technical Services, Manufacturing and Retail Trade. In Professional Services, women now earn 97 cents for every dollar a man earns.

The rest of the industry samples are too small to make meaningful conclusions. Colorado has about 50 observations per month across all workers, and fewer within industries.

> The trend is consistent across urban and rural Colorado.

Due to sample size restrictions, this finding took the simplest measure of the gender wage gap and found that, with no controls, the difference in earnings between men and women shrank by a similar amount in urban (12.2 percentage points) and rural (11.1 points, significant at the 10% level but not the 5% level) Colorado after EPEWA came into effect.

### Results compared with the July 2024 analysis

| Measure | July 2024 (through May 2024) | October 2026 (through September 2026) |
|---|---|---|
| Colorado, change in women's earnings as % of men's (no industry / industry) | 7.7 / 7.5 points | 9.0 / 8.3 points |
| Washington | 5.4 / 5.9 | 6.2 / 6.2 |
| California | 3.2 / 3.3 | 2.7 / 2.7 |
| Rest of the US | 2.0 / 2.0 | 1.8 / 1.8 |
| Colorado dollar effect (industry / no industry) | $3,304 / $3,358 | $3,935 / $4,264 |
| Colorado men's relative change | 1.6 points (p = 0.32) | 2.5 points (p = 0.053) |
| States with a significant change at 5% (no industry / industry) | 14 / 16 | 20 / 20 |

### Limitations of this update

- The 2025 minimum wages come from Wikipedia, and the 2026 wages from a July 2026 snapshot. No official 2025 source exists at the moment. How much this matters has not been tested yet, and a sensitivity check is planned.
- The 34 industry codes introduced by the CPS in 2025 are bridged to sectors using the nearest lower code, because IPUMS has not published a crosswalk for them.
- The October 2025 sample does not exist because the data were not collected during the federal government shutdown.
- Several states in the comparison group enacted their own pay transparency laws in 2023 to 2026, which likely understates the effect for Colorado relative to a clean control group. The extended analysis of the 14 states is planned to address this.